# Experimento Paralelo — 8 workers
## Notebook 05 — Version completa y autonoma

Este notebook genera los YAMLs, crea los sandboxes, y corre el experimento.
Solo necesitas ajustar `BASE_PATH` y `EXPERIMENT_MODE` en la Celda 1.

- `EXPERIMENT_MODE = 'EXP1'` -> SW/HW/GR x DP5/8/10
- `EXPERIMENT_MODE = 'EXP2'` -> desacople fBO4 x S/G (DP=10 fijo)

In [1]:
import os, subprocess, shutil, time, glob, csv, threading, random, yaml
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# ═══════════════════════════════════════════════════════════
# AJUSTA SOLO ESTAS DOS VARIABLES
BASE_PATH       = "/media/aldo/Aldo/sims/Lignin_DB_Natural_Evol_2"
EXPERIMENT_MODE = 'EXP2'   # 'EXP1' o 'EXP2'
N_WORKERS       = 8
N_REPLICAS      = 30
# ═══════════════════════════════════════════════════════════

JAR_PATH      = os.path.join(BASE_PATH, "lgs_gen.jar")
RESOURCES_DIR = os.path.join(BASE_PATH, "resources")
WORKERS_DIR   = os.path.join(BASE_PATH, "workers")

assert os.path.exists(JAR_PATH), f"lgs_gen.jar no encontrado en {JAR_PATH}"
os.makedirs(RESOURCES_DIR, exist_ok=True)

print(f"BASE_PATH:  {BASE_PATH}")
print(f"Modo:       {EXPERIMENT_MODE}")
print(f"Workers:    {N_WORKERS}")
print(f"Replicas:   {N_REPLICAS}")
print("OK")


BASE_PATH:  /media/aldo/Aldo/sims/Lignin_DB_Natural_Evol_2
Modo:       EXP2
Workers:    8
Replicas:   30
OK


In [2]:
# ============================================================
# GENERA LOS 9 YAMLS DIRECTAMENTE EN resources/
# No necesitas descargar ningun ZIP externo
# ============================================================

def write_yaml(path, bondconfig, dp, sg_ratio):
    config = {
        "bondconfig": {
            "B5":   bondconfig["B5"],
            "BB":   bondconfig["BB"],
            "BO4":  bondconfig["BO4"],
            "DBDO": 0,
            "_4O5": bondconfig["_4O5"],
            "_55":  bondconfig["_55"],
        },
        "dp": dp,
        "json": True,
        "matrices": True,
        "monoconfig": {"G": 0, "H": 0, "S": 0},
        "png": True,
        "s_g_ratio": sg_ratio,
        "sdf": True,
    }
    with open(path, "w") as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=True)

# Bondconfigs por tipo botanico (normalizados a 100, valores de literatura MWL)
BONDS = {
    "SW": {"BO4": 54, "B5": 11, "BB": 3, "_4O5": 7, "_55": 25},  # fBO4=0.54
    "HW": {"BO4": 69, "B5":  7, "BB": 7, "_4O5": 7, "_55": 10},  # fBO4=0.69
    "GR": {"BO4": 77, "B5":  5, "BB": 4, "_4O5": 8, "_55":  6},  # fBO4=0.77
}

if EXPERIMENT_MODE == 'EXP1':
    # 3 grupos x 3 niveles de DP
    specs = [
        ("SW_DP5_config.yaml",  "SW", 5,  0.0),
        ("SW_DP8_config.yaml",  "SW", 8,  0.0),
        ("SW_DP10_config.yaml", "SW", 10, 0.0),
        ("HW_DP5_config.yaml",  "HW", 5,  1.5),
        ("HW_DP8_config.yaml",  "HW", 8,  1.5),
        ("HW_DP10_config.yaml", "HW", 10, 1.5),
        ("GR_DP5_config.yaml",  "GR", 5,  0.8),
        ("GR_DP8_config.yaml",  "GR", 8,  0.8),
        ("GR_DP10_config.yaml", "GR", 10, 0.8),
    ]
    CONFIG_META = {
        'SW_DP5':  {'group':'SW','dp':5, 'fBO4':0.54,'sg_ratio':0.0},
        'SW_DP8':  {'group':'SW','dp':8, 'fBO4':0.54,'sg_ratio':0.0},
        'SW_DP10': {'group':'SW','dp':10,'fBO4':0.54,'sg_ratio':0.0},
        'HW_DP5':  {'group':'HW','dp':5, 'fBO4':0.69,'sg_ratio':1.5},
        'HW_DP8':  {'group':'HW','dp':8, 'fBO4':0.69,'sg_ratio':1.5},
        'HW_DP10': {'group':'HW','dp':10,'fBO4':0.69,'sg_ratio':1.5},
        'GR_DP5':  {'group':'GR','dp':5, 'fBO4':0.77,'sg_ratio':0.8},
        'GR_DP8':  {'group':'GR','dp':8, 'fBO4':0.77,'sg_ratio':0.8},
        'GR_DP10': {'group':'GR','dp':10,'fBO4':0.77,'sg_ratio':0.8},
    }
    RESULTS_CSV = os.path.join(BASE_PATH, "exp1_parallel_results.csv")
    CSV_FIELDS  = ['config','group','dp','fBO4','sg_ratio',
                   'replica','t_gen','success','error','timestamp']
else:
    # 3 niveles de fBO4 x 3 niveles de S/G, DP=10 fijo
    specs = [
        ("SW_fBO4_sg0_config.yaml",  "SW", 10, 0.0),
        ("SW_fBO4_sg15_config.yaml", "SW", 10, 1.5),
        ("SW_fBO4_sg08_config.yaml", "SW", 10, 0.8),
        ("HW_fBO4_sg0_config.yaml",  "HW", 10, 0.0),
        ("HW_fBO4_sg15_config.yaml", "HW", 10, 1.5),
        ("HW_fBO4_sg08_config.yaml", "HW", 10, 0.8),
        ("GR_fBO4_sg0_config.yaml",  "GR", 10, 0.0),
        ("GR_fBO4_sg15_config.yaml", "GR", 10, 1.5),
        ("GR_fBO4_sg08_config.yaml", "GR", 10, 0.8),
    ]
    CONFIG_META = {
        'SW_fBO4_sg0':  {'fbo4_source':'SW','fBO4':0.54,'sg_ratio':0.0,'dp':10},
        'SW_fBO4_sg15': {'fbo4_source':'SW','fBO4':0.54,'sg_ratio':1.5,'dp':10},
        'SW_fBO4_sg08': {'fbo4_source':'SW','fBO4':0.54,'sg_ratio':0.8,'dp':10},
        'HW_fBO4_sg0':  {'fbo4_source':'HW','fBO4':0.69,'sg_ratio':0.0,'dp':10},
        'HW_fBO4_sg15': {'fbo4_source':'HW','fBO4':0.69,'sg_ratio':1.5,'dp':10},
        'HW_fBO4_sg08': {'fbo4_source':'HW','fBO4':0.69,'sg_ratio':0.8,'dp':10},
        'GR_fBO4_sg0':  {'fbo4_source':'GR','fBO4':0.77,'sg_ratio':0.0,'dp':10},
        'GR_fBO4_sg15': {'fbo4_source':'GR','fBO4':0.77,'sg_ratio':1.5,'dp':10},
        'GR_fBO4_sg08': {'fbo4_source':'GR','fBO4':0.77,'sg_ratio':0.8,'dp':10},
    }
    RESULTS_CSV = os.path.join(BASE_PATH, "exp2_parallel_results.csv")
    CSV_FIELDS  = ['config','fbo4_source','fBO4','sg_ratio','dp',
                   'replica','t_gen','success','error','timestamp']

# Escribir los 9 YAMLs en resources/
print(f"Generando YAMLs en {RESOURCES_DIR}:")
for fname, bond_key, dp, sg in specs:
    fpath = os.path.join(RESOURCES_DIR, fname)
    write_yaml(fpath, BONDS[bond_key], dp, sg)
    fBO4 = BONDS[bond_key]["BO4"] / 100
    print(f"  {fname:<35} fBO4={fBO4:.2f}  S/G={sg}  DP={dp}")

# Cargar lista de configs
EXPERIMENT_CONFIGS = [os.path.join(RESOURCES_DIR, s[0]) for s in specs]
print(f"\n{len(EXPERIMENT_CONFIGS)} YAMLs listos.")
print(f"Total de corridas: {len(EXPERIMENT_CONFIGS) * N_REPLICAS}")


Generando YAMLs en /media/aldo/Aldo/sims/Lignin_DB_Natural_Evol_2/resources:
  SW_fBO4_sg0_config.yaml             fBO4=0.54  S/G=0.0  DP=10
  SW_fBO4_sg15_config.yaml            fBO4=0.54  S/G=1.5  DP=10
  SW_fBO4_sg08_config.yaml            fBO4=0.54  S/G=0.8  DP=10
  HW_fBO4_sg0_config.yaml             fBO4=0.69  S/G=0.0  DP=10
  HW_fBO4_sg15_config.yaml            fBO4=0.69  S/G=1.5  DP=10
  HW_fBO4_sg08_config.yaml            fBO4=0.69  S/G=0.8  DP=10
  GR_fBO4_sg0_config.yaml             fBO4=0.77  S/G=0.0  DP=10
  GR_fBO4_sg15_config.yaml            fBO4=0.77  S/G=1.5  DP=10
  GR_fBO4_sg08_config.yaml            fBO4=0.77  S/G=0.8  DP=10

9 YAMLs listos.
Total de corridas: 270


In [3]:
def setup_workers(base_path, jar_path, n_workers):
    worker_paths = []
    for i in range(n_workers):
        wp = os.path.join(base_path, "workers", f"worker_{i}")
        os.makedirs(os.path.join(wp, "resources"), exist_ok=True)
        os.makedirs(os.path.join(wp, "output"),    exist_ok=True)
        jar_link = os.path.join(wp, "lgs_gen.jar")
        if not os.path.exists(jar_link):
            os.symlink(jar_path, jar_link)
        worker_paths.append(wp)
    return worker_paths

WORKER_PATHS = setup_workers(BASE_PATH, JAR_PATH, N_WORKERS)
print(f"{N_WORKERS} sandboxes listos")


8 sandboxes listos


In [4]:
csv_lock = threading.Lock()

def run_lgs_in_sandbox(worker_path, config_file, replica_num):
    jar_path    = os.path.join(worker_path, "lgs_gen.jar")
    project_cfg = os.path.join(worker_path, "resources", "project_config.yaml")
    output_dir  = os.path.join(worker_path, "output")
    config_name = os.path.basename(config_file).replace('_config.yaml', '')

    t_gen, success, error_msg = None, False, ""
    try:
        shutil.copy2(config_file, project_cfg)
        t_start = time.time()
        result  = subprocess.run(
            ["java", "-jar", jar_path],
            capture_output=True, text=True, cwd=worker_path
        )
        t_gen   = time.time() - t_start
        success = (result.returncode == 0)
        if not success:
            error_msg = result.stderr[:200]
    except Exception as e:
        error_msg = str(e)
    finally:
        for folder in ['json', 'mol', 'struct', 'matrices']:
            p = os.path.join(output_dir, folder)
            if os.path.exists(p):
                shutil.rmtree(p)
        if os.path.exists(project_cfg):
            os.remove(project_cfg)

    return {'config': config_name, 'replica': replica_num,
            't_gen': round(t_gen, 4) if t_gen else None,
            'success': success, 'error': error_msg,
            'timestamp': datetime.now().isoformat()}

print("Funcion definida")


Funcion definida


In [5]:
print("Prueba rapida: 2 replicas en paralelo...")
with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [
        pool.submit(run_lgs_in_sandbox, WORKER_PATHS[0], EXPERIMENT_CONFIGS[0], 'A'),
        pool.submit(run_lgs_in_sandbox, WORKER_PATHS[1], EXPERIMENT_CONFIGS[0], 'B'),
    ]
    test_results = [f.result() for f in as_completed(futures)]

ok = all(r['success'] for r in test_results)
for r in test_results:
    print(f"  [{'OK' if r['success'] else 'FAIL'}] rep={r['replica']}  "
          f"t_gen={r['t_gen']}s  config={r['config']}")

print("\nSandboxing OK — corre la Celda 6" if ok else
      "\nERROR — revisa BASE_PATH antes de continuar")


Prueba rapida: 2 replicas en paralelo...
  [OK] rep=A  t_gen=14.7566s  config=SW_fBO4_sg0
  [OK] rep=B  t_gen=15.5564s  config=SW_fBO4_sg0

Sandboxing OK — corre la Celda 6


In [6]:
tasks = []
for config_file in EXPERIMENT_CONFIGS:
    for rep in range(1, N_REPLICAS + 1):
        tasks.append((WORKER_PATHS[len(tasks) % N_WORKERS], config_file, rep))
random.shuffle(tasks)

csv_exists = os.path.exists(RESULTS_CSV)
csv_fh     = open(RESULTS_CSV, 'a', newline='')
writer     = csv.DictWriter(csv_fh, fieldnames=CSV_FIELDS)
if not csv_exists:
    writer.writeheader()

done_count = [0]; success_count = [0]
total_tasks = len(tasks); t0 = time.time()

def run_task(args):
    worker_path, config_file, rep = args
    config_name = os.path.basename(config_file).replace('_config.yaml', '')
    meta = CONFIG_META.get(config_name, {})
    r    = run_lgs_in_sandbox(worker_path, config_file, rep)
    row  = {**meta, 'config': r['config'], 'replica': r['replica'],
            't_gen': r['t_gen'] if r['t_gen'] else '',
            'success': r['success'], 'error': r['error'],
            'timestamp': r['timestamp']}
    with csv_lock:
        writer.writerow(row)
        csv_fh.flush()
        done_count[0] += 1; success_count[0] += int(r['success'])
        elapsed = time.time() - t0
        eta = (elapsed / done_count[0]) * (total_tasks - done_count[0])
        print(f"\r  [{done_count[0]:>3}/{total_tasks}] "
              f"{done_count[0]/total_tasks*100:.0f}%  "
              f"ok={success_count[0]}  "
              f"elapsed={elapsed/60:.1f}m  eta={eta/60:.1f}m  "
              f"| {r['config']} rep{rep} {r['t_gen']}s",
              end='', flush=True)
    return r

print(f"EXPERIMENTO PARALELO — {EXPERIMENT_MODE}")
print(f"  Workers:{N_WORKERS}  Tareas:{total_tasks}  CSV:{RESULTS_CSV}")
print("-" * 60)

with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futures = [pool.submit(run_task, t) for t in tasks]
    for _ in as_completed(futures):
        pass

csv_fh.close()
t_total = time.time() - t0
print(f"\n\nCOMPLETO  |  {t_total/60:.1f} min  |  {success_count[0]}/{total_tasks} OK")
print(f"CSV: {RESULTS_CSV}")


EXPERIMENTO PARALELO — EXP2
  Workers:8  Tareas:270  CSV:/media/aldo/Aldo/sims/Lignin_DB_Natural_Evol_2/exp2_parallel_results.csv
------------------------------------------------------------
  [244/270] 90%  ok=239  elapsed=95.4m  eta=10.2m  | HW_fBO4_sg15 rep14 163.9928s

COMPLETO  |  95.4 min  |  239/270 OK
CSV: /media/aldo/Aldo/sims/Lignin_DB_Natural_Evol_2/exp2_parallel_results.csv


In [8]:
from collections import defaultdict

data = defaultdict(list)
with open(RESULTS_CSV) as f:
    for row in csv.DictReader(f):
        if row['success'] == 'True' and row['t_gen']:
            if EXPERIMENT_MODE == 'EXP1':
                key = (row['group'], int(row['dp']))
            else:
                key = (float(row['fBO4']), float(row['sg_ratio']))
            data[key].append(float(row['t_gen']))

def stats(vals):
    n=len(vals); m=sum(vals)/n
    s=(sum((x-m)**2 for x in vals)/n)**0.5
    return n,m,s,min(vals),max(vals)

if EXPERIMENT_MODE == 'EXP1':
    print(f"{'Grupo':<6} {'DP':>4} {'n':>4} {'Media (s)':>10} {'SD':>7}")
    print("-" * 36)
    for g in ['SW','HW','GR']:
        for dp in [5, 8, 10]:
            vals = data.get((g, dp), [])
            if vals:
                n,m,s,mn,mx = stats(vals)
                print(f"{g:<6} {dp:>4} {n:>4} {m:>10.2f} {s:>7.2f}")
else:
    print(f"TABLA t_gen (s) — filas=fBO4, columnas=S/G")
    print(f"{'':>12}  {'S/G=0.0':>10}  {'S/G=0.8':>10}  {'S/G=1.5':>10}")
    print("-" * 48)
    for fbo4, label in [(0.54,'SW'),(0.69,'HW'),(0.77,'GR')]:
        row = f"fBO4={fbo4}({label})  "
        for sg in [0.0, 0.8, 1.5]:
            vals = data.get((fbo4, sg), [])
            m = sum(vals)/len(vals) if vals else float('nan')
            row += f"{m:>10.2f}  "
        print(row)

    print("\nEFECTO PRINCIPAL de fBO4:")
    ranges_fbo4 = []
    for fbo4, label in [(0.54,'SW'),(0.69,'HW'),(0.77,'GR')]:
        all_v = [v for sg in [0.0,0.8,1.5] for v in data.get((fbo4,sg),[])]
        m = sum(all_v)/len(all_v) if all_v else 0
        ranges_fbo4.append(m)
        print(f"  fBO4={fbo4}({label}): {m:.2f}s")

    print("\nEFECTO PRINCIPAL de S/G:")
    ranges_sg = []
    for sg, label in [(0.0,'SW'),(0.8,'GR'),(1.5,'HW')]:
        all_v = [v for fbo4 in [0.54,0.69,0.77] for v in data.get((fbo4,sg),[])]
        m = sum(all_v)/len(all_v) if all_v else 0
        ranges_sg.append(m)
        print(f"  S/G={sg}({label}): {m:.2f}s")

    r_fbo4 = max(ranges_fbo4) - min(ranges_fbo4)
    r_sg   = max(ranges_sg)   - min(ranges_sg)
    print(f"\nRango fBO4: {r_fbo4:.2f}s  |  Rango S/G: {r_sg:.2f}s")
    driver = 'S/G' if r_sg > r_fbo4 else 'fBO4'
    print(f"=> Driver dominante: {driver}")


TABLA t_gen (s) — filas=fBO4, columnas=S/G
                 S/G=0.0     S/G=0.8     S/G=1.5
------------------------------------------------
fBO4=0.54(SW)      100.83      116.45      108.46  
fBO4=0.69(HW)      185.37      374.30      288.86  
fBO4=0.77(GR)       72.46      152.32      140.60  

EFECTO PRINCIPAL de fBO4:
  fBO4=0.54(SW): 108.68s
  fBO4=0.69(HW): 287.70s
  fBO4=0.77(GR): 120.77s

EFECTO PRINCIPAL de S/G:
  S/G=0.0(SW): 117.02s
  S/G=0.8(GR): 216.65s
  S/G=1.5(HW): 179.79s

Rango fBO4: 179.03s  |  Rango S/G: 99.63s
=> Driver dominante: fBO4
